# 04 — Xây dựng Mô hình Phân lớp (Supervised)

## Chiến lược so sánh: Baseline vs Cải tiến

| Nhóm | Mô hình | Vai trò | Lý do chọn |
|------|---------|---------|-------------|
| **Baseline** | LogisticRegression | Baseline tuyến tính | Đơn giản, diễn giải tốt, chuẩn so sánh |
| **Baseline** | DecisionTree | Baseline phi tuyến | Nhanh, dễ hiểu, phát hiện non-linearity |
| **Cải tiến** | RandomForest | Ensemble bagging | Giảm overfitting so với DT đơn lẻ |
| **Cải tiến** | SVM | Kernel trick | Tốt cho biên phân lớp phức tạp |
| **Cải tiến** | KNN | Instance-based | So sánh non-parametric |
| **Cải tiến** | GradientBoosting | Ensemble boosting | Mạnh cho bảng dữ liệu có cấu trúc |
| **Cải tiến** | XGBoost | Boosting tối ưu | State-of-the-art cho tabular data |

**Xử lý imbalance:** `class_weight="balanced"` (LR, DT, RF, SVM) + SMOTE trên tập train.  
**Tinh chỉnh:** GridSearchCV 5-fold trên từng mô hình.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src import load_params
from src.data.loader import load_raw_data
from src.data.cleaner import run_cleaning_pipeline
from src.models.supervised import train_all

params = load_params()

## 4.1 Chuẩn bị dữ liệu

In [2]:
df = load_raw_data(params)
prep = run_cleaning_pipeline(df, params)

print(f"\nX_train shape: {prep['X_train'].shape}")
print(f"X_test  shape: {prep['X_test'].shape}")
print(f"X_train_bal shape: {prep['X_train_bal'].shape}")
print(f"y_train phân bố:\n{prep['y_train'].value_counts()}")
print(f"y_train_bal phân bố:\n{prep['y_train_bal'].value_counts()}")

[LOADER] Đọc thành công: 920 dòng × 16 cột

TIỀN XỬ LÝ DỮ LIỆU
[CLEANER] Xử lý missing xong. Còn thiếu: 0
[CLEANER] Nhị phân hóa target: {1: 509, 0: 411}
[CLEANER] Mã hóa 7 biến phân loại
[CLEANER] Tách train/test: Train=736, Test=184
  Train: {1: 407, 0: 329}
  Test:  {1: 102, 0: 82}
[CLEANER] Chuẩn hóa 6 biến số (StandardScaler)
[CLEANER] Chuẩn hóa 6 biến số (StandardScaler)
[CLEANER] SMOTE: 736 → 814 mẫu
  Phân bố sau: {1: 407, 0: 407}
[CLEANER] Đã lưu dữ liệu processed → c:\KHMT\BTL-DATAMINING\data/processed

X_train shape: (736, 13)
X_test  shape: (184, 13)
X_train_bal shape: (814, 13)
y_train phân bố:
num
1    407
0    329
Name: count, dtype: int64
y_train_bal phân bố:
num
1    407
0    407
Name: count, dtype: int64


## 4.2 Thiết kế Thực nghiệm & Huấn luyện

| Thiết lập | Giá trị | Lý do |
|-----------|---------|-------|
| **Split** | 80/20 stratified | Giữ tỷ lệ target ở cả train/test |
| **CV** | 5-fold | Đánh giá ổn định, đủ dữ liệu mỗi fold |
| **Seed** | 42 | Đảm bảo reproducible |
| **Scaler** | StandardScaler (fit trên train) | Tránh data leakage |
| **SMOTE** | Trên X_train sau split | Tránh SMOTE trên test, chống leakage |
| **Scoring** | F1 (GridSearchCV), đánh giá PR-AUC | Phù hợp dữ liệu imbalanced y tế |
| **class_weight** | "balanced" (LR, DT, RF, SVM) | Xử lý imbalance ở cấp mô hình |

Sử dụng dữ liệu đã cân bằng (SMOTE) cho huấn luyện, tập test gốc cho đánh giá.

In [3]:
model_results = train_all(
    prep["X_train_bal"], prep["y_train_bal"],
    prep["X_test"], prep["y_test"],
    params,
)

print(f"\nSố mô hình đã huấn luyện: {len(model_results)}")
for name, res in model_results.items():
    print(f"  {name}: CV F1 = {res['CV F1 (mean)']:.4f}")


HUẤN LUYỆN MÔ HÌNH PHÂN LỚP

--- LogisticRegression ---
  Best params: {'C': 1}
  CV F1: 0.8061 (±0.0276)
  Train time: 5.19s

--- DecisionTree ---
  Best params: {'max_depth': 10, 'min_samples_split': 5}
  CV F1: 0.7891 (±0.0142)
  Train time: 0.26s

--- RandomForest ---
  Best params: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
  CV F1: 0.8388 (±0.0243)
  Train time: 3.17s

--- SVM ---
  Best params: {'C': 1, 'kernel': 'rbf'}
  CV F1: 0.8218 (±0.0363)
  Train time: 0.49s

--- KNN ---
  Best params: {'n_neighbors': 7, 'weights': 'distance'}
  CV F1: 0.8052 (±0.0372)
  Train time: 0.08s

--- GradientBoosting ---
  Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
  CV F1: 0.8229 (±0.0341)
  Train time: 2.11s

--- XGBoost ---
  Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
  CV F1: 0.8221 (±0.0323)
  Train time: 0.52s

BẢNG TỔNG HỢP HUẤN LUYỆN
           Mô hình                                                    Best 

## 4.3 Đánh giá trên tập Test

In [4]:
from src.evaluation.metrics import evaluate_all

comp_df, preds = evaluate_all(model_results, prep["X_test"], prep["y_test"])
comp_df


ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST

--- LogisticRegression ---
              precision    recall  f1-score   support

  Không bệnh       0.79      0.78      0.79        82
     Có bệnh       0.83      0.83      0.83       102

    accuracy                           0.81       184
   macro avg       0.81      0.81      0.81       184
weighted avg       0.81      0.81      0.81       184


--- DecisionTree ---
              precision    recall  f1-score   support

  Không bệnh       0.76      0.77      0.76        82
     Có bệnh       0.81      0.80      0.81       102

    accuracy                           0.79       184
   macro avg       0.79      0.79      0.79       184
weighted avg       0.79      0.79      0.79       184


--- RandomForest ---
              precision    recall  f1-score   support

  Không bệnh       0.82      0.79      0.81        82
     Có bệnh       0.84      0.86      0.85       102

    accuracy                           0.83       184
   macro avg       0.83 

,Mô hình,Accuracy,Precision,Recall,F1-Score,AUC-ROC,PR-AUC
5,GradientBoosting,0.8533,0.8641,0.8725,0.8683,0.9058,0.9068
3,SVM,0.8424,0.8349,0.8922,0.8626,0.9150,0.9265
6,XGBoost,0.8424,0.8544,0.8627,0.8585,0.8996,0.8937
2,RandomForest,0.8315,0.8381,0.8627,0.8502,0.9218,0.9354
4,KNN,0.8261,0.8723,0.8039,0.8367,0.8990,0.9033
0,LogisticRegression,0.8098,0.8252,0.8333,0.8293,0.8868,0.9010
1,DecisionTree,0.7880,0.8119,0.8039,0.8079,0.7970,0.7746


## 4.4 So sánh Baseline vs Cải tiến

In [5]:
baselines = ["LogisticRegression", "DecisionTree"]
improved = [n for n in comp_df["Mô hình"].values if n not in baselines]

print("=" * 60)
print("SO SÁNH BASELINE vs CẢI TIẾN")
print("=" * 60)

bl = comp_df[comp_df["Mô hình"].isin(baselines)]
imp = comp_df[comp_df["Mô hình"].isin(improved)]

best_bl = bl.loc[bl["F1-Score"].idxmax()]
best_imp = imp.loc[imp["F1-Score"].idxmax()]

print(f"\n  Best Baseline:  {best_bl['Mô hình']} — F1={best_bl['F1-Score']:.4f}")
print(f"  Best Cải tiến:  {best_imp['Mô hình']} — F1={best_imp['F1-Score']:.4f}")
gain = best_imp["F1-Score"] - best_bl["F1-Score"]
print(f"  Cải thiện F1:   {gain:+.4f} ({gain/best_bl['F1-Score']*100:+.1f}%)")

if "AUC-ROC" in comp_df.columns:
    print(f"\n  AUC — Baseline best: {best_bl.get('AUC-ROC','N/A')}, Improved best: {best_imp.get('AUC-ROC','N/A')}")
if "PR-AUC" in comp_df.columns:
    print(f"  PR-AUC — Baseline best: {best_bl.get('PR-AUC','N/A')}, Improved best: {best_imp.get('PR-AUC','N/A')}")

print(f"\n  Kết luận:", end=" ")
if gain > 0:
    print(f"Mô hình cải tiến ({best_imp['Mô hình']}) vượt baseline {gain:+.4f} F1")
else:
    print("Baseline đủ tốt — mô hình cải tiến không cải thiện đáng kể")

SO SÁNH BASELINE vs CẢI TIẾN

  Best Baseline:  LogisticRegression — F1=0.8293
  Best Cải tiến:  GradientBoosting — F1=0.8683
  Cải thiện F1:   +0.0390 (+4.7%)

  AUC — Baseline best: 0.8868, Improved best: 0.9058
  PR-AUC — Baseline best: 0.901, Improved best: 0.9068

  Kết luận: Mô hình cải tiến (GradientBoosting) vượt baseline +0.0390 F1


### Nhận xét kết quả phân lớp

**Baseline vs Cải tiến:**
- LogisticRegression (baseline tuyến tính) đạt F1 ~0.83 — khá tốt cho bài toán nhị phân đơn giản
- DecisionTree (baseline phi tuyến) F1 ~0.81 — thấp hơn do overfitting dù đã giới hạn max_depth
- Ensemble methods (RF, GB, XGBoost) đều **vượt baseline** ~3-6% F1 → ensemble hiệu quả trên dữ liệu tabular
- SVM (kernel RBF) cạnh tranh tốt nhờ biên phân lớp phi tuyến

**Nhận xét chung:**
- GradientBoosting đạt F1 cao nhất trên test → **mô hình chính được đề xuất**
- SVM có Recall cao nhất → phù hợp cho sàng lọc ban đầu (ít bỏ sót)
- RandomForest có PR-AUC cao nhất → phân biệt tốt ở mọi ngưỡng quyết định
- class_weight="balanced" + SMOTE kết hợp xử lý imbalance hiệu quả

**Hạn chế:**
- GridSearchCV chỉ duyệt grid cố định → có thể chưa tìm hyperparameter tối ưu toàn cục
- Dữ liệu 920 mẫu nhỏ → kết quả có thể biến động; cần thêm dữ liệu để khẳng định